# Stateless runtime

**Objective:** Run replaceable web processes that keep durable state in backing services and start or stop cleanly.

## Simple version

A Twelve-Factor process owns temporary computation only; durable or shared state belongs in an attached service.

In [ ]:
# Process memory is temporary; durable state belongs in an attached service.
process = {
    "bind": "0.0.0.0:8000",
    "local_session_state": None,
    "durable_state": "database or Redis",
}

print(process)

## Polished version

Two replaceable FastAPI processes share an injected backing store. Lifespan hooks open and close resources, while the runtime supplies the bound port.

In [ ]:
from collections.abc import AsyncIterator
from contextlib import asynccontextmanager
from typing import Protocol

import httpx
from fastapi import FastAPI


# The web process depends on a store contract, not a specific database.
class CounterStore(Protocol):
    async def connect(self) -> None: ...
    async def close(self) -> None: ...
    async def increment(self) -> int: ...


class MemoryAttachedStore:
    """Standalone substitute for PostgreSQL or Redis."""

    def __init__(self) -> None:
        self.connections = 0
        self.value = 0

    async def connect(self) -> None:
        self.connections += 1

    async def close(self) -> None:
        self.connections -= 1

    async def increment(self) -> int:
        if self.connections < 1:
            raise RuntimeError("backing service is not connected")
        self.value += 1
        return self.value


def create_app(store: CounterStore) -> FastAPI:
    @asynccontextmanager
    async def lifespan(app: FastAPI) -> AsyncIterator[None]:
        # Open and close external connections once per process lifecycle.
        await store.connect()
        yield
        await store.close()

    app = FastAPI(lifespan=lifespan)

    @app.post("/visits")
    async def record_visit() -> dict:
        return {"visits": await store.increment()}

    return app


def web_command(port: int) -> list[str]:
    return [
        "uvicorn",
        "app:app",
        "--host",
        "0.0.0.0",
        "--port",
        str(port),
    ]


# These represent two replaceable web processes sharing one external store.
store = MemoryAttachedStore()
first_app = create_app(store)
second_app = create_app(store)

async with (
    first_app.router.lifespan_context(first_app),
    second_app.router.lifespan_context(second_app),
):
    first_transport = httpx.ASGITransport(app=first_app)
    second_transport = httpx.ASGITransport(app=second_app)

    async with (
        httpx.AsyncClient(
            transport=first_transport,
            base_url="http://first",
        ) as first_client,
        httpx.AsyncClient(
            transport=second_transport,
            base_url="http://second",
        ) as second_client,
    ):
        first = await first_client.post("/visits")
        second = await second_client.post("/visits")

print(first.json(), second.json())
print("Run:", " ".join(web_command(8000)))
print("Open connections after shutdown:", store.connections)

## Applied in this repository

Both project app factories manage clients through FastAPI lifespan hooks. PostgreSQL and Redis hold shared state, Uvicorn binds the web port, and the LLM worker scales independently from the web process.